### RAG Pipelines - Data Ingestion to Vector DB pipeline

In [ ]:
import os 
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
def process_all_pdfs(pdf_dir):
    """Process all PDF files in the specified directory and return a list of documents with metadata."""
    all_documents = []
    pdf_dir_path = Path(pdf_dir)

    pdf_files = list(pdf_dir_path.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process.")

    for pdf_file in pdf_files:
        print(f"\nProcessing {pdf_file}...")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source metadata to each document, additional metadata can be added here as needed
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = 'pdf'

            all_documents.extend(documents)
            print(f"\nLoaded {len(documents)} documents from {pdf_file}.")

        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_docs = process_all_pdfs("../data/pdf")
print(f"\nTotal documents processed: {len(all_docs)}")

In [ ]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks using RecursiveCharacterTextSplitter for RAG processing."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    if split_docs:
        print(f"\nSample split document metadata: {split_docs[0].metadata}")
        print(f"\nSample split document content (first 200 chars): {split_docs[0].page_content[:200]}")
    return split_docs

In [ ]:
chunked_docs = split_documents(all_docs)
print(f"\nTotal chunked documents: {len(chunked_docs)}")

### Embedding and VectorstoreDB

In [ ]:
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    """Manages embedding generation using SentenceTransformer and storage/retrieval in ChromaDB."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager.
        
        Args:
            model_name (str): The name of the HuggingFace model to use for embeddings.
            """
        
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """Load the SentenceTransformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts.
        
        Args:
            texts (List[str]): A list of strings to generate embeddings for.
        Returns:
            np.ndarray: An array of embeddings corresponding to the input texts.   
        """

        if not self.model:
            raise ValueError("Embedding model is not loaded.") 
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

# Initialize the embedding manager and generate embeddings for the chunked documents
embedding_manager = EmbeddingManager()

## VectorStore 

In [ ]:
class VectorStore:
    """Manages vector storage and retrieval using ChromaDB."""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """Initialize the vector store.
        
        Args:
            collection_name (str): The name of the ChromaDB collection to use for storing vectors.
            persist_directory (str): The directory where ChromaDB will persist its data.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_chromadb()
    
    def _initialize_chromadb(self):
        """Initialize the ChromaDB client and collection."""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"descrption": "Collection for storing PDF document embeddings for RAG pipelines"}
            )
            print(f"ChromaDB collection '{self.collection_name}' initialized successfully at '{self.persist_directory}'.")
            print(f"Existing documents in collection: {len(self.collection.get(ids=None))}")
        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise

    def add_documents(self, documents: List[Dict[str, Any]], embeddings: np.ndarray):
        """Add documents to the vector store with generated embeddings.
        
        Args:
            documents: List of documents to add, where each document is a dict with 'page_content' and 'metadata'.
            embeddings: List of embeddings corresponding to the documents.
        """
        
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents and embeddings must be the same.")
        
        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []

        for i, (doc, emb) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            documents_texts.append(doc.page_content)
            embeddings_list.append(emb.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_texts
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection after addition: {len(self.collection.get(ids=None))}")
        
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


vector_store = VectorStore()
vector_store

In [ ]:
## Generate embeddings for the chunked documents
chunked_texts = [doc.page_content for doc in chunked_docs]
chunked_embeddings = embedding_manager.generate_embeddings(chunked_texts)
vector_store.add_documents(chunked_docs, chunked_embeddings)

### Retriever Pipeline From VectorStoreDB:

In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 / (1 + distance)
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vector_store, embedding_manager)


In [110]:
rag_retriever.retrieve("What is attention is all you need?")

Retrieving documents for query: 'What is attention is all you need?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.92it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_a0fb27fe_32',
  'content': '3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3',
  'metadata': {'creator': 'LaTeX with hyperref',
   'author': '',
   'title': '',
   'keywords': '',
   'creationdate': '2024-04-10T21:11:43+00:00',
   'total_pages': 15,
   'producer': 'pdfTeX-1.40.25',
   'doc_index': 32,
   'file_type': 'pdf',
   'trapped': '/False',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'source_file': '1706.03762v7.pdf',
   'subject': '',
   'moddate': '2024-04-10T21:11:43+00:00',
   'page': 2,
   'source': '../data/pdf/1706.03762v7.pdf',
   'content_length': 216,
   'page_label': '3'},
  'similarity_score': 0.5346515179452513,
  'distance': 0.8703771829605103,
  'rank': 1},
 {'id': 'doc_ed4f89b1_12',
  'content': '3.2 Attention\nA